# 📜 Lab 3: The Royal Bard — Train a Transformer to Spin Fairy Tales

> *"Once upon a time in a faraway kingdom..."* — Your model writes the rest. ✨

---

**OBJECTIVE:** You already trained an **RNN** to generate text in Lab 2 (the Tech-Fluencer). Now you'll do the **same thing** with a **Transformer** — the architecture behind ChatGPT.

**THE SCENARIO:** A children's bedtime-story app needs an AI bard to spin endless fairy tales. The catch: their old RNN bard kept forgetting the beginning of long stories. Your job: replace it with a Transformer that can *attend* to the entire context at once.

---

## 🧠 What's the same as the RNN lab?

| RNN lab (Lab 2) | This lab |
|---|---|
| `string.punctuation` cleaning | ✅ same |
| Keras `Tokenizer` | ✅ same |
| `pad_sequences(..., padding='pre')` | ✅ same |
| N-gram sequences `[:i+1]` | ✅ same |
| `Dense(VOCAB_SIZE, softmax)` output | ✅ same |
| `sparse_categorical_crossentropy` + `adam` | ✅ same |
| Temperature-sampled generation loop | ✅ same |

## 🆕 What's new? (just the model)

Everything around the model is identical — only **Part 5** changes:

| RNN model (Lab 2) | Transformer model (this lab) |
|---|---|
| `LSTM(150)` | `MultiHeadAttention` + position embeddings + FFN |
| `Sequential` API | **Functional** API (because attention has residual connections) |

You'll plug in the **exact same building blocks** you learned in the **PyTorch transformer lecture** — embeddings → Q/K/V attention → FFN → softmax — just expressed in Keras.

## 🎮 Self-grading

Every TODO has points (🟢 5 / 🟡 10 / 🔴 20), totalling **110 points** across 11 tasks. Run the **`# 🎯 CHECK`** cell after each TODO. Climb the rank ladder:

- 🥚 **Egg in the Nest** (0–25%)
- 🐣 **Wandering Apprentice** (25–50%)
- 📖 **Village Storyteller** (50–70%)
- 🎭 **Royal Court Bard** (70–85%)
- 🪄 **Master Wordweaver** (85–99%)
- 👑 **Eternal Bard of the Realm** (100%)

📌 Hints live in `<details>` blocks — open them only after you've genuinely tried.


## Part 0: Setup & Data 🟢 *(provided — no points, just run it)*

In [1]:
# Standard imports — same stack as Lab 2 + a couple of extras for the transformer
import string
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Embedding, Dense, Dropout, Layer,
    LayerNormalization, MultiHeadAttention, Lambda
)

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)
print("GPU available:", bool(tf.config.list_physical_devices('GPU')))


TensorFlow version: 2.20.0
GPU available: False


### 🎮 Initialize the scoreboard

Run this cell once. It creates a `grader` object that tracks your points all the way through. Don't re-run later — it'll reset your score!

In [2]:
class RoyalBardGrader:
    """Self-grading system for the Royal Bard transformer lab."""

    RANKS = [
        (0.00, "🥚 Egg in the Nest",            "Cozy and quiet. The story has yet to begin."),
        (0.25, "🐣 Wandering Apprentice",       "You've left the nest. The path is foggy but yours."),
        (0.50, "📖 Village Storyteller",        "Children gather at your feet. Your tales spread."),
        (0.70, "🎭 Royal Court Bard",           "The king summons you. Your verses fill the great hall."),
        (0.85, "🪄 Master Wordweaver",          "Your name is whispered across kingdoms. Legends abound."),
        (1.00, "👑 Eternal Bard of the Realm",  "Your tales never end. The realm tells them forever."),
    ]

    def __init__(self, total_possible=110):
        self.scores = {}
        self.max_points = {}
        self.total_possible = total_possible

    @property
    def total_earned(self):
        return sum(self.scores.values())

    def _bar(self, pct, width=30):
        filled = int(round(pct * width))
        return "█" * filled + "░" * (width - filled)

    def _rank(self):
        pct = self.total_earned / self.total_possible if self.total_possible else 0
        rank = self.RANKS[0]
        for threshold, name, blurb in self.RANKS:
            if pct >= threshold:
                rank = (threshold, name, blurb)
        return rank

    def check(self, task_id, passed, points, success_msg="", fail_msg="", hint=""):
        self.max_points[task_id] = points
        already_passed = self.scores.get(task_id, 0) == points

        if passed:
            self.scores[task_id] = points
            badge = "🔁 Already complete" if already_passed else f"✅ +{points} points!"
            print(f"{badge}  [Task {task_id}]")
            if success_msg and not already_passed:
                print(f"   {success_msg}")
        else:
            if not already_passed:
                self.scores[task_id] = 0
            print(f"❌ Task {task_id} not passing yet. (0 / {points} points)")
            if fail_msg:
                print(f"   {fail_msg}")
            if hint:
                print(f"   💡 Hint: {hint}")

        self._show()

    def _show(self):
        pct = self.total_earned / self.total_possible if self.total_possible else 0
        _, rank_name, rank_blurb = self._rank()
        print()
        print(f"   Score:  {self.total_earned} / {self.total_possible}  [{self._bar(pct)}] {pct*100:.0f}%")
        print(f"   Rank:   {rank_name}")
        print(f"           {rank_blurb}")

    def scorecard(self):
        pct = self.total_earned / self.total_possible if self.total_possible else 0
        _, rank_name, rank_blurb = self._rank()
        print("=" * 56)
        print("🏆  FINAL SCORECARD")
        print("=" * 56)
        for tid in sorted(self.max_points.keys()):
            earned = self.scores.get(tid, 0)
            total  = self.max_points[tid]
            mark   = "✅" if earned == total else ("🟡" if earned > 0 else "❌")
            print(f"  {mark}  Task {tid:<8s}  {earned:>3d} / {total:<3d}")
        print("-" * 56)
        print(f"  TOTAL:               {self.total_earned:>3d} / {self.total_possible}")
        print(f"  [{self._bar(pct)}] {pct*100:.0f}%")
        print(f"  Rank: {rank_name}")
        print(f"        {rank_blurb}")
        print("=" * 56)


grader = RoyalBardGrader(total_possible=110)
print("🎮 Scoreboard initialized. Total points up for grabs: 110")
print("   Earn them by completing the TODOs in each Part.")


🎮 Scoreboard initialized. Total points up for grabs: 110
   Earn them by completing the TODOs in each Part.


### 📚 Get the dataset

No download needed — we ship with **50 fairy-tale snippets** baked into the notebook. The corpus is small *on purpose*: the transformer should train in 1–3 minutes on CPU and ≤1 minute on GPU.

If you want a bigger corpus later, drop a `tales.csv` file with a `text` column next to this notebook and the loader will pick it up automatically.

In [3]:
import os

# Try a real file first; fall back to the built-in 50-tale corpus.
df = None
for path in ["tales.csv", "data/tales.csv"]:
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"✅ Loaded real dataset from {path} — {len(df)} rows")
        break

if df is None:
    print("⚠️  Real dataset not found — using built-in sample of 50 fairy-tale snippets.")
    SAMPLE_TALES = [
        "Once upon a time in a faraway kingdom there lived a kind prince who loved to wander the forest at dawn looking for adventure.",
        "Long long ago in a tiny village by the silver river a clever girl dreamed of seeing the wide ocean and sailing to distant lands.",
        "Deep in the enchanted forest a brave knight stumbled upon a sleeping dragon coiled around a golden apple tree.",
        "In a hidden valley between two snowy mountains the wise old owl kept the secrets of every star in the sky.",
        "There was once a kindhearted baker who shared his last loaf of bread with a beggar and was rewarded with three magical wishes.",
        "Beyond the misty hills lived a young witch who could speak to flowers and her best friend was a curious silver fox.",
        "In a great stone castle on the cliff the lonely princess sang every night to the moon hoping someone would hear her song.",
        "Once in a forest of tall whispering trees a little wolf cub got lost and met a friendly squirrel who knew every path home.",
        "Long ago a brave farmer climbed the giant beanstalk to find the cloud kingdom where the wind giants lived in towers of gold.",
        "In the heart of the desert there was a hidden oasis where a wise old camel guarded a fountain of starlight.",
        "There once was a humble cobbler who made boots so beautiful that even the queen of the fairies came to buy a pair.",
        "Long ago in a cottage by the woods a kind grandmother taught her grandchild how to bake pies that could mend a broken heart.",
        "In a kingdom by the sea three brothers set out on a quest to find the singing fish that lived beneath the waves.",
        "Once upon a time a tiny mouse with a giant heart befriended the king of the lions and saved his life with a single nibble.",
        "Deep in the enchanted forest a clever fox tricked a hungry bear and earned the friendship of every creature in the woods.",
        "In the mountains of mist a young shepherd boy played a flute so sweet that the stars came down to listen.",
        "There was once a princess who refused to marry any prince and instead rode off on her unicorn to slay her own dragons.",
        "Long long ago a kind old fisherman caught a golden fish that granted him three wishes if he would set it free.",
        "In a quiet village at the edge of the woods a curious child found a door in the trunk of an ancient oak tree.",
        "Once upon a time the moon fell in love with the sea and every night they whispered secrets that no one else could hear.",
        "Beyond the dark forest there stood a tower of silver where a clever girl was kept by a witch with eyes like emeralds.",
        "Long ago a young wizard set out on a journey to find the lost spell that would bring color back to a gray kingdom.",
        "In a forgotten kingdom of stone a sleeping queen waited for the day a true friend would wake her with a song of hope.",
        "Once upon a time a small dragon was afraid of fire and so he spent his days reading books in a quiet cave.",
        "Deep in the valley of flowers a tiny fairy lost her wings and a kind little boy helped her find them again.",
        "In a faraway land a brave knight rode on his loyal horse to rescue a wise old wizard from a tower of thorns.",
        "Long long ago in a town under the stars there lived a baker who could make cakes that gave you sweet dreams.",
        "Once in a kingdom of glass mountains a clever shepherdess outwitted three trolls and freed the trapped princes inside.",
        "There was once a magical rabbit who kept the time of the forest and every animal trusted his small silver pocket watch.",
        "In a cottage made of candy a kind witch baked cookies for lost children and helped them find their way back home.",
        "Long ago a brave little tailor met a giant in the meadow and won his friendship with a pot of strawberry jam.",
        "Once upon a time a young prince traded his crown for a chance to live as an ordinary person and find true love.",
        "In the deepest part of the ocean a kind mermaid kept a garden of glowing pearls that lit the darkness for sailors above.",
        "Beyond the river of stars a tiny fairy painted the colors of the sunrise every morning with a brush made of moonlight.",
        "There once was a magical horse with a mane of fire who carried a brave girl across a kingdom of ice and snow.",
        "Long long ago a kindhearted shoemaker was helped by tiny elves who finished his work each night while he slept peacefully.",
        "In a kingdom of singing rivers a young bard played a harp that could turn enemies into friends with a single note.",
        "Once upon a time a clever queen invited every dragon to a feast and they all left as her loyal allies forever.",
        "Deep in the woods a small lost duckling was raised by a family of foxes who taught her how to be brave and wise.",
        "In a town where it always snowed a tiny child dreamed of summer and a kind fairy granted her one warm day.",
        "Long ago a brave knight discovered that the dragon he was sent to slay was actually a princess under a dark spell.",
        "Once upon a time a humble gardener found a magic seed that grew into a beanstalk reaching to the kingdom of the clouds.",
        "In a dark cave a curious young witch met a friendly bat who taught her the songs of the night creatures.",
        "There was once a tiny prince no bigger than a thumb who rode on the back of a sparrow to faraway lands.",
        "Long long ago a kind old woman planted a single rose that grew into a forest of red blooms that healed broken hearts.",
        "In a magical land of endless summer a wise old turtle told stories to every child who came to sit by his pond.",
        "Once upon a time a brave little dog led a lost prince through the dark forest back to his worried family in the castle.",
        "Beyond the icy mountains a young hunter befriended a snow leopard and together they protected the secret of the silver lake.",
        "Deep in the valley of stars a kind shepherd boy gave his cloak to a freezing fairy and was given the gift of flight.",
        "In a faraway kingdom by the sea a clever fisher girl outsmarted a sea serpent and brought peace to her village forever.",
    ]
    df = pd.DataFrame({"text": SAMPLE_TALES})
    print(f"✅ Sample dataset loaded — {len(df)} fairy-tale snippets")

# Find the text column flexibly
text_col = next((c for c in ["text", "Text", "tale", "story", "content"] if c in df.columns), None)
if text_col is None:
    raise ValueError(f"No text column found. Available: {list(df.columns)}")

print(f"Using column: '{text_col}'")
df = df.dropna(subset=[text_col]).copy()
print(f"Final size: {len(df)} tales")
df[text_col].head(3)


⚠️  Real dataset not found — using built-in sample of 50 fairy-tale snippets.
✅ Sample dataset loaded — 50 fairy-tale snippets
Using column: 'text'
Final size: 50 tales


,text
0,Once upon a time in a faraway kingdom there li...
1,Long long ago in a tiny village by the silver ...
2,Deep in the enchanted forest a brave knight st...


## Part 1: Noise Removal & Normalization 🟡

This is **identical to Part 1 of the RNN lab**: lowercase + strip punctuation using `string.punctuation`. We just apply it to a different corpus.

In [4]:
# 🟡 TODO 1.1  (10 points)
# Write a clean_text(text) function that:
#   (a) Converts text to lowercase
#   (b) Removes all punctuation using string.punctuation
#   (c) Returns the result as a string
#
# This is the EXACT same function from Lab 2. Copy it here.

def clean_text(text):
    text = str(text).lower()
    # YOUR CODE HERE
    lower_case=text.lower()
    translator=str.maketrans("","",string.punctuation)
    return text.translate(translator)

    pass

# Quick visual test
sample = "Once Upon A Time, In A KINGDOM Far, Far Away... There lived a Brave Prince!"
print("Original:", sample)
print("Cleaned :", clean_text(sample))


Original: Once Upon A Time, In A KINGDOM Far, Far Away... There lived a Brave Prince!
Cleaned : once upon a time in a kingdom far far away there lived a brave prince


<details>
<summary>💡 Hint 1.1 — same as Lab 2 Part 1</summary>

```python
def clean_text(text):
    text = str(text).lower()
    translator = str.maketrans('', '', string.punctuation)
    return text.translate(translator)
```
</details>

In [5]:
# 🎯 CHECK 1.1
try:
    out = clean_text("Once Upon A Time, In A KINGDOM Far, Far Away!")
    expected_words = {"once", "upon", "a", "time", "in", "kingdom", "far", "away"}
    actual_words = set(out.split())
    no_punct = all(ch not in out for ch in string.punctuation)
    is_lower = out == out.lower()
    passed = expected_words.issubset(actual_words) and no_punct and is_lower
    grader.check("1.1", passed, 10,
                 success_msg="Cleaning works! Lowercase + punctuation stripped.",
                 fail_msg=f"Got: {out!r}",
                 hint="Use text.lower() and str.maketrans('', '', string.punctuation).")
except Exception as e:
    grader.check("1.1", False, 10, fail_msg=f"Error: {e}",
                 hint="Make sure clean_text is defined and returns a string.")


✅ +10 points!  [Task 1.1]
   Cleaning works! Lowercase + punctuation stripped.

   Score:  10 / 110  [███░░░░░░░░░░░░░░░░░░░░░░░░░░░] 9%
   Rank:   🥚 Egg in the Nest
           Cozy and quiet. The story has yet to begin.


In [6]:
# Apply cleaning to the whole dataset (provided)
df["clean"] = df[text_col].apply(clean_text)

# Drop tales that are too short (we need at least a few words for n-gram pairs)
word_counts = df["clean"].str.split().str.len()
df = df[word_counts >= 8].reset_index(drop=True)

print(f"After cleaning + filtering: {len(df)} tales")
print("\nExample cleaned tale:")
print(df["clean"].iloc[0])


After cleaning + filtering: 50 tales

Example cleaned tale:
once upon a time in a faraway kingdom there lived a kind prince who loved to wander the forest at dawn looking for adventure


## Part 2: Tokenizer & Vocabulary 🟢

**Identical to Part 2 of the RNN lab.** Same `Tokenizer`, same `texts_to_sequences`. Just feeding it fairy tales instead of LinkedIn rants.

In [7]:
# 🟢 TODO 2.1  (5 points)
# Create and fit a Tokenizer on df["clean"].
# - Use vocab_size = 2000  (smaller than Lab 2 because the corpus is smaller)
# - Use oov_token = "<OOV>"
# - Then call .fit_on_texts() on the cleaned tales.

VOCAB_SIZE=2000

# YOUR CODE HERE
tokenizer = Tokenizer(num_words=VOCAB_SIZE,oov_token="<OOV>")

tokenizer.fit_on_texts(df["clean"])

<details>
<summary>💡 Hint 2.1</summary>

```python
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(sentences)
```
</details>

In [8]:
# 🎯 CHECK 2.1
try:
    has_word_index = hasattr(tokenizer, "word_index") and len(tokenizer.word_index) > 50
    has_oov = "<OOV>" in tokenizer.word_index
    grader.check("2.1", has_word_index and has_oov, 5,
                 success_msg=f"Tokenizer fit on {len(tokenizer.word_index):,} unique words. "
                             f"Top 5: {list(tokenizer.word_index.items())[:5]}",
                 fail_msg="Tokenizer not fit correctly.",
                 hint="tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>'); "
                      "tokenizer.fit_on_texts(sentences)")
except NameError:
    grader.check("2.1", False, 5, fail_msg="`tokenizer` is not defined.",
                 hint="Did you actually create the Tokenizer object?")


✅ +5 points!  [Task 2.1]
   Tokenizer fit on 409 unique words. Top 5: [('<OOV>', 1), ('a', 2), ('the', 3), ('of', 4), ('in', 5)]

   Score:  15 / 110  [████░░░░░░░░░░░░░░░░░░░░░░░░░░] 14%
   Rank:   🥚 Egg in the Nest
           Cozy and quiet. The story has yet to begin.


In [9]:
# 🟢 TODO 2.2  (5 points)
# Convert each tale in `sentences` to a list of integer IDs using texts_to_sequences.
# Same as Lab 2.

# YOUR CODE HERE
input_sequences_per_tale = tokenizer.texts_to_sequences(df["clean"])

# Sanity peek (uncomment after filling in)
print("First 15 IDs of tale 0:", input_sequences_per_tale[0][:15])
print("That tale in words:  ", df["clean"][0][:80], "...")


First 15 IDs of tale 0: [8, 19, 2, 20, 5, 2, 40, 16, 17, 33, 2, 21, 34, 11, 148]
That tale in words:   once upon a time in a faraway kingdom there lived a kind prince who loved to wan ...


<details>
<summary>💡 Hint 2.2</summary>

```python
input_sequences_per_tale = tokenizer.texts_to_sequences(sentences)
```
</details>

In [10]:
# 🎯 CHECK 2.2
try:
    is_list = isinstance(input_sequences_per_tale, list)
    nonempty = len(input_sequences_per_tale) == len(df["clean"])
    has_ints = isinstance(input_sequences_per_tale[0][0], int)
    grader.check("2.2", is_list and nonempty and has_ints, 5,
                 success_msg=f"Got {len(input_sequences_per_tale)} sequences of integer IDs.",
                 fail_msg="Output should be a list of lists of ints.",
                 hint="input_sequences_per_tale = tokenizer.texts_to_sequences(sentences)")
except NameError:
    grader.check("2.2", False, 5, fail_msg="`input_sequences_per_tale` is not defined.")


✅ +5 points!  [Task 2.2]
   Got 50 sequences of integer IDs.

   Score:  20 / 110  [█████░░░░░░░░░░░░░░░░░░░░░░░░░] 18%
   Rank:   🥚 Egg in the Nest
           Cozy and quiet. The story has yet to begin.


## Part 3: N-Gram Sequences 🔴 *(same trick as Lab 2)*

The transformer learns the same way the RNN did: from `(seen-so-far → next word)` pairs. We build them with the same n-gram trick.

| Input (what it has seen) | Target (what comes next) |
|---|---|
| `[10]` | `7` |
| `[10, 7]` | `22` |
| `[10, 7, 22]` | `5` |
| `[10, 7, 22, 5]` | `91` |

We keep input + target glued together (last element = target) and split them after padding in Part 4.

In [11]:
# 🔴 TODO 3.1  (20 points) — same loop as Lab 2.
# Build an `input_sequences` list. For each tokenized tale in `input_sequences_per_tale`,
# create n-gram sequences of length 2, 3, 4, ..., len(tale).
#
# Example:
#   tale = [10, 7, 22, 5, 91]
#   should add to input_sequences:
#     [10, 7]
#     [10, 7, 22]
#     [10, 7, 22, 5]
#     [10, 7, 22, 5, 91]

input_sequences = []

# YOUR CODE HERE  (use a nested for loop)
for token_list in input_sequences_per_tale:
  for i in range(1,len(token_list)):
    n_gram=token_list[:i+1]
    input_sequences.append(n_gram)






print(f"Generated {len(input_sequences):,} n-gram training sequences from {len(input_sequences_per_tale)} tales.")
print(f"First 3 n-grams from tale 0: {input_sequences[:3]}")


Generated 1,079 n-gram training sequences from 50 tales.
First 3 n-grams from tale 0: [[8, 19], [8, 19, 2], [8, 19, 2, 20]]


<details>
<summary>💡 Hint 3.1 — exactly the same as Lab 2</summary>

```python
input_sequences = []
for token_list in input_sequences_per_tale:
    for i in range(1, len(token_list)):
        n_gram = token_list[: i + 1]
        input_sequences.append(n_gram)
```
</details>

In [12]:
# 🎯 CHECK 3.1
try:
    is_list = isinstance(input_sequences, list)
    not_empty = len(input_sequences) > 100
    has_pair = any(len(s) == 2 for s in input_sequences)
    has_triple = any(len(s) == 3 for s in input_sequences)
    well_formed = all(isinstance(s, list) and all(isinstance(t, int) for t in s)
                      for s in input_sequences[:50])
    passed = is_list and not_empty and has_pair and has_triple and well_formed
    grader.check("3.1", passed, 20,
                 success_msg=f"Built {len(input_sequences):,} n-grams. "
                             f"Lengths range {min(len(s) for s in input_sequences)}–"
                             f"{max(len(s) for s in input_sequences)}.",
                 fail_msg="N-gram list looks off.",
                 hint="for token_list in input_sequences_per_tale: "
                      "for i in range(1, len(token_list)): "
                      "input_sequences.append(token_list[:i+1])")
except NameError:
    grader.check("3.1", False, 20, fail_msg="`input_sequences` is not defined.")


✅ +20 points!  [Task 3.1]
   Built 1,079 n-grams. Lengths range 2–25.

   Score:  40 / 110  [███████████░░░░░░░░░░░░░░░░░░░] 36%
   Rank:   🐣 Wandering Apprentice
           You've left the nest. The path is foggy but yours.


## Part 4: Padding & X / y Split 🟡

**Same as Lab 2.** Pre-pad with zeros so the *target* sits on the right edge of every row, then peel off the last column as `y`.

```
[0, 0, 0, 10, 22, 5]   ← padding='pre'
                  ↑ target lives here
```

In [13]:
# 🟡 TODO 4.1  (10 points) — same as Lab 2.
# 1. Find the length of the longest n-gram (we'll pad everything to this length).
# 2. Use pad_sequences with padding='pre' to convert input_sequences -> a numpy array.

# YOUR CODE HERE
max_seq_len = max(len(x) for x in input_sequences)
padded = np.array(pad_sequences(input_sequences,maxlen=max_seq_len,padding="pre"))

print(f"Max sequence length: {max_seq_len}")
print(f"Padded shape:        {padded.shape}")
print(f"Example padded row:  {padded[0]}")


Max sequence length: 25
Padded shape:        (1079, 25)
Example padded row:  [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  8
 19]


<details>
<summary>💡 Hint 4.1</summary>

```python
max_seq_len = max(len(x) for x in input_sequences)
padded = np.array(pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre'))
```
</details>

In [14]:
# 🎯 CHECK 4.1
try:
    correct_shape = padded.shape == (len(input_sequences), max_seq_len)
    is_pre_padded = padded[0].tolist().count(0) > 0
    grader.check("4.1", correct_shape and is_pre_padded, 10,
                 success_msg=f"Padded matrix shape {padded.shape}, pre-padded with zeros on the left.",
                 fail_msg=f"Padding shape: {padded.shape}",
                 hint="pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre')")
except NameError:
    grader.check("4.1", False, 10, fail_msg="`padded` or `max_seq_len` not defined.")


✅ +10 points!  [Task 4.1]
   Padded matrix shape (1079, 25), pre-padded with zeros on the left.

   Score:  50 / 110  [██████████████░░░░░░░░░░░░░░░░] 45%
   Rank:   🐣 Wandering Apprentice
           You've left the nest. The path is foggy but yours.


In [15]:
# 🟢 TODO 4.2  (5 points) — same as Lab 2.
# Split each padded row:
#   X = everything except the last column        -> the input to the transformer
#   y = the last column only                     -> the target word ID we want to predict

# YOUR CODE HERE
X = padded[:,:-1]
y = padded[:,-1]

# print(f"X shape: {X.shape}   (input sequences, length {max_seq_len - 1})")
# print(f"y shape: {y.shape}   (one target ID per row)")
# print(f"Example: X[0] = {X[0]}")
# print(f"         y[0] = {y[0]}")


<details>
<summary>💡 Hint 4.2</summary>

```python
X = padded[:, :-1]
y = padded[:, -1]
```
</details>

In [16]:
# 🎯 CHECK 4.2
try:
    correct_X = X.shape == (len(input_sequences), max_seq_len - 1)
    correct_y = y.shape == (len(input_sequences),)
    grader.check("4.2", correct_X and correct_y, 5,
                 success_msg=f"X is {X.shape}, y is {y.shape}. Ready to train.",
                 fail_msg=f"Got X={X.shape}, y={y.shape}",
                 hint="X = padded[:, :-1]; y = padded[:, -1]")
except NameError:
    grader.check("4.2", False, 5, fail_msg="X or y not defined.")


✅ +5 points!  [Task 4.2]
   X is (1079, 24), y is (1079,). Ready to train.

   Score:  55 / 110  [███████████████░░░░░░░░░░░░░░░] 50%
   Rank:   📖 Village Storyteller
           Children gather at your feet. Your tales spread.


## Part 5: Build the Transformer 🟡 *(NEW — but it's just the PyTorch lecture in Keras)*

This is the **only structural change** from Lab 2. Instead of `LSTM(150)` we use:
- a **token + positional embedding** (so the model knows *which* word AND *where* it is)
- a **transformer block** = self-attention + FFN, with residual connections + LayerNorm
- a **final Dense softmax** (same as Lab 2)

### 🧠 What the transformer block does (review from the PyTorch lecture)

```
        ┌──────────────────────────────────────────┐
input → │ Multi-Head Self-Attention (causal mask)  │ → + → LayerNorm → ┐
        └──────────────────────────────────────────┘   ↑               │
                                                       └── residual ───┘
                                                                       │
                                                                       ▼
                                       ┌──────────────────────────┐
                                       │ Feed-Forward Network     │ → + → LayerNorm → output
                                       │   Linear → ReLU → Linear │   ↑
                                       └──────────────────────────┘   └── residual
```

The two helper classes below implement this. **Read them — don't write them** (they wrap exactly what you saw in the lecture). You'll just *call* them in TODO 5.1.

In [17]:
# Provided helper #1 — Token + Positional Embedding
# This is the only NEW idea vs. Lab 2: transformers don't see word order automatically,
# so we ADD a "position embedding" to each token embedding.

class TokenAndPositionEmbedding(Layer):
    def __init__(self, max_len, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_emb = Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.pos_emb   = Embedding(input_dim=max_len,   output_dim=embed_dim)

    def call(self, x):
        positions = tf.range(start=0, limit=tf.shape(x)[-1], delta=1)
        return self.token_emb(x) + self.pos_emb(positions)


# Provided helper #2 — One Transformer block
# This is EXACTLY the diagram above: attention → residual+norm → FFN → residual+norm.
# Compare line-by-line with the PyTorch lecture: it's the same math, different framework.

class TransformerBlock(Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        # Multi-head self-attention (the heart of the transformer)
        self.attn = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        # Feed-Forward Network: Linear -> ReLU -> Linear (just like the lecture)
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim),
        ])
        self.norm1, self.norm2 = LayerNormalization(), LayerNormalization()
        self.drop1, self.drop2 = Dropout(dropout), Dropout(dropout)

    def call(self, x, training=False):
        # Self-attention with CAUSAL mask (each position only sees earlier positions)
        attn_out = self.attn(x, x, use_causal_mask=True)
        attn_out = self.drop1(attn_out, training=training)
        x = self.norm1(x + attn_out)             # residual + LayerNorm

        # Feed-Forward Network
        ffn_out = self.ffn(x)
        ffn_out = self.drop2(ffn_out, training=training)
        return self.norm2(x + ffn_out)            # residual + LayerNorm


print("✅ Helper classes defined. Now you just have to wire them up.")


✅ Helper classes defined. Now you just have to wire them up.


In [18]:
# Hyperparameters (provided)
VOCAB_SIZE_REAL = min(len(tokenizer.word_index) + 1, VOCAB_SIZE)
EMBED_DIM = 64    # size of each word's vector (= d_model from the lecture)
NUM_HEADS = 4     # number of attention heads
FF_DIM    = 128   # hidden size inside the FFN (typically 2-4x EMBED_DIM)
SEQ_LEN   = X.shape[1]

print(f"Real vocab size: {VOCAB_SIZE_REAL}")
print(f"Sequence length: {SEQ_LEN}")
print(f"Embedding dim:   {EMBED_DIM}")


Real vocab size: 410
Sequence length: 24
Embedding dim:   64


In [19]:
# 🟡 TODO 5.1  (10 points) — Assemble the transformer model.
# Fill in the THREE blanks below using the helpers above.

inputs = Input(shape=(SEQ_LEN,))

# (a) Token + Positional embeddings  -- replaces the Embedding(...) layer from Lab 2.
# TODO: call TokenAndPositionEmbedding on `inputs`.
# Hint: TokenAndPositionEmbedding(SEQ_LEN, VOCAB_SIZE_REAL, EMBED_DIM)(inputs)
x = TokenAndPositionEmbedding(SEQ_LEN, VOCAB_SIZE_REAL, EMBED_DIM)(inputs)

# (b) One transformer block  -- replaces the LSTM(150) layer from Lab 2.
# TODO: call TransformerBlock on `x`.
# Hint: TransformerBlock(EMBED_DIM, NUM_HEADS, FF_DIM)(x)
x = TransformerBlock(EMBED_DIM, NUM_HEADS, FF_DIM)(x)

# (provided) Take the LAST token's representation -- that's where the next-word prediction lives.
# (With pre-padding, position -1 is the most recent context word.)
x = Lambda(lambda t: t[:, -1, :])(x)

# (c) Final softmax over the vocab -- IDENTICAL to Lab 2's Dense(VOCAB_SIZE_REAL, softmax).
# TODO: a Dense layer mapping to VOCAB_SIZE_REAL with softmax.
# Hint: Dense(VOCAB_SIZE_REAL, activation="softmax")(x)
output = Dense(VOCAB_SIZE_REAL, activation="softmax")(x)

model = Model(inputs, output)
model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 24)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding    │ (None, 24, 64)         │        27,776 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block               │ (None, 24, 64)         │        83,200 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 410)            │        26,650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 137,626 (537.60 KB)

 Trainable params: 137,626 (537.60 KB)

 Non-trainable params: 0 (0.00 B)

<details>
<summary>💡 Hint 5.1 — full assembly</summary>

```python
inputs = Input(shape=(SEQ_LEN,))
x = TokenAndPositionEmbedding(SEQ_LEN, VOCAB_SIZE_REAL, EMBED_DIM)(inputs)
x = TransformerBlock(EMBED_DIM, NUM_HEADS, FF_DIM)(x)
x = Lambda(lambda t: t[:, -1, :])(x)
output = Dense(VOCAB_SIZE_REAL, activation="softmax")(x)
model = Model(inputs, output)
```

🔍 **What just happened?** You implemented the same architecture as the PyTorch lecture, in Keras. Token IDs go in → become vectors → attention lets each position look at every earlier position → FFN refines → take the last position → predict the next word.
</details>

In [20]:
# 🎯 CHECK 5.1
try:
    layer_types = [type(layer).__name__ for layer in model.layers]
    has_embedding = "TokenAndPositionEmbedding" in layer_types
    has_transformer = "TransformerBlock" in layer_types
    last_layer = model.layers[-1]
    last_units = getattr(last_layer, "units", None)
    correct_output = last_units == VOCAB_SIZE_REAL
    passed = has_embedding and has_transformer and correct_output
    grader.check("5.1", passed, 10,
                 success_msg=f"Model built: {' → '.join(layer_types)}. "
                             f"Output dim {last_units} = vocab size. ✓",
                 fail_msg=f"Layers: {layer_types}, last Dense units: {last_units}",
                 hint="Need: TokenAndPositionEmbedding → TransformerBlock → Lambda(last token) → Dense(vocab, softmax)")
except NameError:
    grader.check("5.1", False, 10, fail_msg="`model` not defined.")


✅ +10 points!  [Task 5.1]
   Model built: InputLayer → TokenAndPositionEmbedding → TransformerBlock → Lambda → Dense. Output dim 410 = vocab size. ✓

   Score:  65 / 110  [██████████████████░░░░░░░░░░░░] 59%
   Rank:   📖 Village Storyteller
           Children gather at your feet. Your tales spread.


## Part 6: Compile & Train 🟡

**Identical to Part 6 of Lab 2.** Same loss, same optimizer, same `model.fit`. The transformer cares about the input shape and the loss — neither has changed.

In [21]:
# 🟡 TODO 6.1  (10 points) — same as Lab 2.
# Compile the model with:
#   - loss = 'sparse_categorical_crossentropy'
#   - optimizer = 'adam'
#   - metrics = ['accuracy']

# YOUR CODE HERE
model.compile(loss="sparse_categorical_crossentropy",optimizer="adam",metrics=["accuracy"])


<details>
<summary>💡 Hint 6.1</summary>

```python
model.compile(loss='sparse_categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])
```
</details>

In [22]:
# 🎯 CHECK 6.1
try:
    loss_name = model.loss if isinstance(model.loss, str) else model.loss.__class__.__name__.lower()
    is_sparse_cce = "sparse_categorical_crossentropy" in str(loss_name).lower()
    grader.check("6.1", is_sparse_cce, 10,
                 success_msg=f"Compiled with sparse_categorical_crossentropy + Adam.",
                 fail_msg=f"Got loss = {loss_name}",
                 hint="loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy']")
except Exception as e:
    grader.check("6.1", False, 10, fail_msg=f"Error checking compile: {e}")


✅ +10 points!  [Task 6.1]
   Compiled with sparse_categorical_crossentropy + Adam.

   Score:  75 / 110  [████████████████████░░░░░░░░░░] 68%
   Rank:   📖 Village Storyteller
           Children gather at your feet. Your tales spread.


In [23]:
# 🟢 TODO 6.2  (5 points) — Train the model.
# Use model.fit on (X, y). Reasonable defaults provided.
# This corpus is small, so 30 epochs is enough.

EPOCHS = 30
BATCH_SIZE = 64

# YOUR CODE HERE
history = model.fit(X,y)


34/34 ━━━━━━━━━━━━━━━━━━━━ 10s 45ms/step - accuracy: 0.0982 - loss: 5.6271


<details>
<summary>💡 Hint 6.2</summary>

```python
history = model.fit(X, y, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=1)
```

📉 You should see loss drop steadily. Accuracy will look low (10–30%) — that's normal for language modeling, because at any position there are many *plausible* next words.
</details>

In [24]:
# 🎯 CHECK 6.2
try:
    has_history = 'history' in dir() and hasattr(history, 'history')
    final_loss = history.history.get('loss', [None])[-1]
    trained = has_history and final_loss is not None and final_loss < 8.0
    grader.check("6.2", trained, 5,
                 success_msg=f"Model trained! Final loss: {final_loss:.4f}",
                 fail_msg="`history` not produced or loss looks wrong.",
                 hint="history = model.fit(X, y, epochs=EPOCHS, batch_size=BATCH_SIZE)")
except Exception as e:
    grader.check("6.2", False, 5, fail_msg=f"Training didn't complete: {e}")


✅ +5 points!  [Task 6.2]
   Model trained! Final loss: 5.6271

   Score:  80 / 110  [██████████████████████░░░░░░░░] 73%
   Rank:   🎭 Royal Court Bard
           The king summons you. Your verses fill the great hall.


## Part 7: Generate Fairy Tales 🔴 *(same loop as Lab 2)*

The generation loop is **literally the same code** as Lab 2 — the model under the hood happens to be a transformer, but the API to call it is identical.

1. Tokenize the seed.
2. Pad it to length `max_seq_len - 1`.
3. Predict the probability distribution over the next word.
4. Apply temperature and sample one word.
5. Append it to the seed. Repeat.

### 🌡️ Temperature reminder
- `T → 0`: greedy, picks the single most likely word every time. Boring, repetitive.
- `T = 1.0`: samples from the model's actual distribution. Balanced.
- `T > 1.0`: flattens the distribution. More random. More creative. More chaos.

In [25]:
# Helper: fast lookup from word ID -> word string. (Build once, use many times.)
index_to_word = {idx: word for word, idx in tokenizer.word_index.items()}
print("Sample lookups:", {1: index_to_word.get(1), 2: index_to_word.get(2), 3: index_to_word.get(3)})


Sample lookups: {1: '<OOV>', 2: 'a', 3: 'the'}


In [26]:
# 🔴 TODO 7.1  (20 points) — Implement the generation loop.
# This is COPY-PASTE from Lab 2. The model just happens to be a transformer now.

def generate_text(seed_text, num_words, model, max_seq_len, temperature=1.0):
    """Generate `num_words` more words after `seed_text`."""
    seed_text = clean_text(seed_text)  # apply same preprocessing as training!

    for _ in range(num_words):
         token_list=tokenizer.texts_to_sequences([seed_text])[0]



        # ---- (b) Pre-pad to length max_seq_len - 1 ----
         token_list = pad_sequences([token_list], maxlen=max_seq_len - 1, padding='pre')


        # ---- (c) Predict the probability distribution over the next word ----
         probs = model.predict(token_list, verbose=0)[0]

        # ---- (d) Apply temperature, then sample ----
        # See hint below.  Use np.random.choice(len(probs), p=probs_after_temperature).
        # YOUR CODE HERE
         predicted_id = np.random.choice(len(probs), p=probs)
        # YOUR CODE HERE...

        # ---- (e) Convert ID back to a word and append ----
         output_word = index_to_word.get(predicted_id, "")
         if output_word == "" or output_word == "<OOV>":
              break
         seed_text += " " + output_word
        # YOUR CODE HERE

         pass  # remove this once you've filled in the steps

    return seed_text


# Quick test (you'll see real output once your code is filled in)
print(generate_text("Once upon a time in a faraway", num_words=20, model=model,
                    max_seq_len=max_seq_len, temperature=1.0))


once upon a time in a faraway would rose mist loved of of a darkness loved mend a through children was to to brave friendship small for


<details>
<summary>💡 Hint 7.1 — full body of the loop (same as Lab 2)</summary>

```python
def generate_text(seed_text, num_words, model, max_seq_len, temperature=1.0):
    seed_text = clean_text(seed_text)
    for _ in range(num_words):
        # (a) tokenize
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        # (b) pad (pre-padding, same as training)
        token_list = pad_sequences([token_list], maxlen=max_seq_len - 1, padding='pre')
        # (c) predict probabilities
        probs = model.predict(token_list, verbose=0)[0]
        # (d) apply temperature and sample
        probs = np.log(probs + 1e-10) / temperature
        probs = np.exp(probs) / np.sum(np.exp(probs))
        predicted_id = np.random.choice(len(probs), p=probs)
        # (e) append the chosen word
        output_word = index_to_word.get(predicted_id, "")
        if output_word == "" or output_word == "<OOV>":
            break
        seed_text += " " + output_word
    return seed_text
```

**Why `np.log(...) / T` then re-softmax?** Dividing log-probs by `T` and re-normalizing is the standard temperature trick. Larger `T` → flatter distribution → more diversity.
</details>

In [27]:
# 🎯 CHECK 7.1
try:
    out = generate_text("Once upon a time in a faraway", num_words=10, model=model,
                        max_seq_len=max_seq_len, temperature=1.0)
    is_str = isinstance(out, str)
    grew = len(out.split()) > len("Once upon a time in a faraway".split())
    grader.check("7.1", is_str and grew, 20,
                 success_msg=f"Generation works! Sample output:\n   '{out}'",
                 fail_msg=f"Generated: {out!r} — looks empty or unchanged.",
                 hint="Check (a) tokenize -> (b) pad -> (c) predict -> (d) sample -> (e) append.")
except Exception as e:
    grader.check("7.1", False, 20, fail_msg=f"generate_text raised: {e}",
                 hint="Open the hint dropdown above for the full loop body.")


✅ +20 points!  [Task 7.1]
   Generation works! Sample output:
   'once upon a time in a faraway coiled the forest mountains helped forgotten and beanstalk dawn a'

   Score:  100 / 110  [███████████████████████████░░░] 91%
   Rank:   🪄 Master Wordweaver
           Your name is whispered across kingdoms. Legends abound.


In [28]:
# 🟡 TODO 7.2  (10 points) — Generate 3 tales at 3 different temperatures.
# Use the SAME seed for all three so you can compare.
# Required: at least one temperature < 1.0 and one > 1.0.

SEED = "Once upon a time in a faraway kingdom there lived"

# YOUR CODE HERE — print 3 generations with different temperatures.
# E.g.: temperatures = [0.4, 1.0, 1.5]
for T in [0.4, 1.0, 1.5]:
  print(generate_text(SEED,num_words=40,model=model,max_seq_len=max_seq_len,temperature=T))



once upon a time in a faraway kingdom there lived loved of a planted sit cakes brave the fountain boy of cottage a duckling kind cloud prince played no and led gray than elves apple warm a his a magical so every cookies three brave love of shepherdess a given
once upon a time in a faraway kingdom there lived fell once a trapped grandmother mist emeralds a little the darkness star could a dragon dragon instead someone elves flowers of hungry stood them eyes castle fish rode a of a someone prince broken allies ago valley edge her a
once upon a time in a faraway kingdom there lived upon the magic outwitted bread above rescue in finished granted trapped slept wise the kingdom boy princes pearls a slay boy her of magical sparrow prince shepherd quest witch hunter stories his how lived by a sky friendly a old


<details>
<summary>💡 Hint 7.2</summary>

```python
for T in [0.4, 1.0, 1.5]:
    print(f"\n--- Temperature {T} ---")
    print(generate_text(SEED, num_words=40, model=model,
                        max_seq_len=max_seq_len, temperature=T))
```

🔍 **What to notice:** at low T the tale gets repetitive (e.g. "and the kind kind kind…"); at high T it gets wild and dreamlike. Around T=0.7–1.0 is the sweet spot for fairy-tale vibes.
</details>

In [29]:
# 🎯 CHECK 7.2 — relaxed; we just verify you tried multiple temperatures and got distinct results.
try:
    out_low  = generate_text(SEED, num_words=15, model=model, max_seq_len=max_seq_len, temperature=0.4)
    out_mid  = generate_text(SEED, num_words=15, model=model, max_seq_len=max_seq_len, temperature=1.0)
    out_high = generate_text(SEED, num_words=15, model=model, max_seq_len=max_seq_len, temperature=1.5)
    distinct = len({out_low, out_mid, out_high}) >= 2
    grader.check("7.2", distinct, 10,
                 success_msg="Different temperatures produce different vibes. Nice work.",
                 fail_msg="All temperatures produced the same text — temperature scaling may be off.",
                 hint="Check the np.log(probs)/T then softmax-renormalize step.")
except Exception as e:
    grader.check("7.2", False, 10, fail_msg=f"Sweep raised: {e}")


✅ +10 points!  [Task 7.2]
   Different temperatures produce different vibes. Nice work.

   Score:  110 / 110  [██████████████████████████████] 100%
   Rank:   👑 Eternal Bard of the Realm
           Your tales never end. The realm tells them forever.


## 🏆 Final Scorecard

Run the cell below to see your full breakdown and unlocked rank.

In [30]:
grader.scorecard()


🏆  FINAL SCORECARD
  ✅  Task 1.1        10 / 10 
  ✅  Task 2.1         5 / 5  
  ✅  Task 2.2         5 / 5  
  ✅  Task 3.1        20 / 20 
  ✅  Task 4.1        10 / 10 
  ✅  Task 4.2         5 / 5  
  ✅  Task 5.1        10 / 10 
  ✅  Task 6.1        10 / 10 
  ✅  Task 6.2         5 / 5  
  ✅  Task 7.1        20 / 20 
  ✅  Task 7.2        10 / 10 
--------------------------------------------------------
  TOTAL:               110 / 110
  [██████████████████████████████] 100%
  Rank: 👑 Eternal Bard of the Realm
        Your tales never end. The realm tells them forever.


## Part 8: Bonus Challenges 🔥 *(beyond 100% — for the show-offs)*

Each bonus you ship is worth pride points (and bragging rights). No automatic grading.

### 🥉 Easy bonuses
- **Stack two transformer blocks.** Just call `TransformerBlock(...)` twice in a row. Compare quality.
- **Train for 60 epochs instead of 30.** When does the loss flatten out?
- **Try `EMBED_DIM = 32` vs `EMBED_DIM = 128`.** Smaller is faster, bigger is more expressive — what's the trade-off?

### 🥈 Medium bonuses
- **Top-k sampling.** Modify `generate_text` so it only samples from the top-k most likely tokens (e.g. k=10) — even at high temperature this keeps the tale coherent.
- **Number of heads.** Try `NUM_HEADS = 1` vs `NUM_HEADS = 8`. Same total parameters but different "perspectives" — does it matter on this small corpus?
- **Use your own corpus.** Drop a `tales.csv` (or any text file with a `text` column) into the folder and re-run. Star Wars scripts? Pirate logs? Your favourite cookbook?

### 🥇 Hard bonuses
- **Compare with Lab 2.** Train the same data on the LSTM from Lab 2 and on this transformer. Plot training loss curves on the same axes. Which converges faster? Which generates better tales at temperature=1?
- **Beam search decoding.** Instead of sampling, keep the top-B partial sequences at each step and pick the highest-probability one at the end.
- **Compare attention patterns.** After training, extract `model.layers[<the TransformerBlock>].attn` weights for a sample input and visualize them as a heatmap. Which words attend to which?

---

## 🎓 What you just built

You implemented every piece of a small **decoder-only transformer language model** — the same architecture (just smaller) that powers GPT-class models:

- **Token + positional embeddings** so the model knows *what* and *where*.
- **Multi-head self-attention** with a **causal mask** — exactly the Q · Kᵀ / √d → softmax → · V you saw in the PyTorch lecture.
- **Residual connections + LayerNorm** so the network trains stably.
- **Feed-Forward Network** to refine each position independently.
- **Cross-entropy + Adam + temperature sampling** — same training and decoding loop you used for the RNN.

The only real difference between this and a tiny GPT is **scale** (more layers, more heads, way more data). Same recipe.

> "And they all decoded happily ever after." 📜


## Bonus Challenge 1: Stack two Transformer Blocks

Here, we'll redefine the model to include two `TransformerBlock` layers sequentially. This increases the model's capacity to learn more complex relationships.

In [31]:
# Redefine hyperparameters (if they were changed elsewhere)
# Use the consistent EMBED_DIM for this stacked model
# EMBED_DIM1 and EMBED_DIM2 are not needed here if stacking blocks of the same dimension
# EMBED_DIM is 64 from previous cells.
# NUM_HEADS = 4
# FF_DIM = 128
SEQ_LEN = X.shape[1]
VOCAB_SIZE_REAL = min(len(tokenizer.word_index) + 1, VOCAB_SIZE)

# Model with two TransformerBlocks using the same EMBED_DIM
inputs_bonus1 = Input(shape=(SEQ_LEN,))
x = TokenAndPositionEmbedding(SEQ_LEN, VOCAB_SIZE_REAL, EMBED_DIM)(inputs_bonus1)
x = TransformerBlock(EMBED_DIM, NUM_HEADS, FF_DIM)(x)
x = TransformerBlock(EMBED_DIM, NUM_HEADS, FF_DIM)(x) # Second Transformer Block
x = Lambda(lambda t: t[:, -1, :])(x)
output_bonus1 = Dense(VOCAB_SIZE_REAL, activation="softmax")(x)

model_bonus1 = Model(inputs_bonus1, output_bonus1)
model_bonus1.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

print("Model with two Transformer Blocks summary:")
model_bonus1.summary()

Model with two Transformer Blocks summary:


Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 24)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding_1  │ (None, 24, 64)         │        27,776 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_1             │ (None, 24, 64)         │        83,200 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_2             │ (None, 24, 64)         │        83,200 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda_1 (Lambda)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 410)            │        26,650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 220,826 (862.60 KB)

 Trainable params: 220,826 (862.60 KB)

 Non-trainable params: 0 (0.00 B)

Now, let's train this model and see how it performs.

In [ ]:
print("Training model with two Transformer Blocks...")
history_bonus1 = model_bonus1.fit(X, y, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=1)

print("\nGenerated text with two Transformer Blocks (Temperature 1.0):")
print(generate_text(SEED, num_words=40, model=model_bonus1, max_seq_len=max_seq_len, temperature=1.0))

Training model with two Transformer Blocks...
Epoch 1/30
17/17 ━━━━━━━━━━━━━━━━━━━━ 11s 166ms/step - accuracy: 0.0908 - loss: 5.6695
Epoch 2/30
17/17 ━━━━━━━━━━━━━━━━━━━━ 3s 164ms/step - accuracy: 0.1233 - loss: 5.1873
Epoch 3/30
17/17 ━━━━━━━━━━━━━━━━━━━━ 6s 190ms/step - accuracy: 0.1603 - loss: 4.7024
Epoch 4/30
17/17 ━━━━━━━━━━━━━━━━━━━━ 3s 163ms/step - accuracy: 0.2345 - loss: 4.1976
Epoch 5/30
17/17 ━━━━━━━━━━━━━━━━━━━━ 3s 161ms/step - accuracy: 0.3262 - loss: 3.7326
Epoch 6/30
17/17 ━━━━━━━━━━━━━━━━━━━━ 3s 181ms/step - accuracy: 0.4078 - loss: 3.3649
Epoch 7/30
15/17 ━━━━━━━━━━━━━━━━━━━━ 0s 234ms/step - accuracy: 0.4465 - loss: 3.1233

## Bonus Challenge 2: Train for 60 Epochs

We'll go back to the original single-block transformer but train it for a longer duration (60 epochs) to see if the loss continues to decrease or flattens out.

In [ ]:
# Redefine hyperparameters (if they were changed elsewhere)
EMBED_DIM = 64
NUM_HEADS = 4
FF_DIM = 128
SEQ_LEN = X.shape[1]
VOCAB_SIZE_REAL = min(len(tokenizer.word_index) + 1, VOCAB_SIZE)

# Original model (single TransformerBlock)
inputs_bonus2 = Input(shape=(SEQ_LEN,))
x = TokenAndPositionEmbedding(SEQ_LEN, VOCAB_SIZE_REAL, EMBED_DIM)(inputs_bonus2)
x = TransformerBlock(EMBED_DIM, NUM_HEADS, FF_DIM)(x)
x = Lambda(lambda t: t[:, -1, :])(x)
output_bonus2 = Dense(VOCAB_SIZE_REAL, activation="softmax")(x)

model_bonus2 = Model(inputs_bonus2, output_bonus2)
model_bonus2.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

print("Model with single Transformer Block for 60 epochs summary:")
model_bonus2.summary()

Now, let's train this model for 60 epochs.

In [ ]:
print("Training model for 60 epochs...")
EXTENDED_EPOCHS = 60
history_bonus2 = model_bonus2.fit(X, y, epochs=EXTENDED_EPOCHS, batch_size=BATCH_SIZE, verbose=1)

print("\nGenerated text after 60 epochs (Temperature 1.0):")
print(generate_text(SEED, num_words=40, model=model_bonus2, max_seq_len=max_seq_len, temperature=1.0))

## Bonus Challenge 3: Try `EMBED_DIM = 32` vs `EMBED_DIM = 128`

Let's investigate the impact of the embedding dimension. A smaller `EMBED_DIM` (32) should be faster but less expressive, while a larger one (128) should be more expressive but slower.

### `EMBED_DIM = 32`

In [ ]:
print("--- Training with EMBED_DIM = 32 ---")

# Redefine hyperparameters
EMBED_DIM_32 = 32
NUM_HEADS = 4
FF_DIM = EMBED_DIM_32 * 2 # Adjust FF_DIM proportionally
SEQ_LEN = X.shape[1]
VOCAB_SIZE_REAL = min(len(tokenizer.word_index) + 1, VOCAB_SIZE)

# Model definition for EMBED_DIM = 32
inputs_bonus3_32 = Input(shape=(SEQ_LEN,))
x = TokenAndPositionEmbedding(SEQ_LEN, VOCAB_SIZE_REAL, EMBED_DIM_32)(inputs_bonus3_32)
x = TransformerBlock(EMBED_DIM_32, NUM_HEADS, FF_DIM)(x)
x = Lambda(lambda t: t[:, -1, :])(x)
output_bonus3_32 = Dense(VOCAB_SIZE_REAL, activation="softmax")(x)

model_bonus3_32 = Model(inputs_bonus3_32, output_bonus3_32)
model_bonus3_32.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

print("Model summary for EMBED_DIM=32:")
model_bonus3_32.summary()

print("Training model with EMBED_DIM = 32...")
history_bonus3_32 = model_bonus3_32.fit(X, y, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=1)

print("\nGenerated text with EMBED_DIM = 32 (Temperature 1.0):")
print(generate_text(SEED, num_words=40, model=model_bonus3_32, max_seq_len=max_seq_len, temperature=1.0))

### `EMBED_DIM = 128`

In [ ]:
print("--- Training with EMBED_DIM = 128 ---")

# Redefine hyperparameters
EMBED_DIM_128 = 128
NUM_HEADS = 4
FF_DIM = EMBED_DIM_128 * 2 # Adjust FF_DIM proportionally
SEQ_LEN = X.shape[1]
VOCAB_SIZE_REAL = min(len(tokenizer.word_index) + 1, VOCAB_SIZE)

# Model definition for EMBED_DIM = 128
inputs_bonus3_128 = Input(shape=(SEQ_LEN,))
x = TokenAndPositionEmbedding(SEQ_LEN, VOCAB_SIZE_REAL, EMBED_DIM_128)(inputs_bonus3_128)
x = TransformerBlock(EMBED_DIM_128, NUM_HEADS, FF_DIM)(x)
x = Lambda(lambda t: t[:, -1, :])(x)
output_bonus3_128 = Dense(VOCAB_SIZE_REAL, activation="softmax")(x)

model_bonus3_128 = Model(inputs_bonus3_128, output_bonus3_128)
model_bonus3_128.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

print("Model summary for EMBED_DIM=128:")
model_bonus3_128.summary()

print("Training model with EMBED_DIM = 128...")
history_bonus3_128 = model_bonus3_128.fit(X, y, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=1)

print("\nGenerated text with EMBED_DIM = 128 (Temperature 1.0):")
print(generate_text(SEED, num_words=40, model=model_bonus3_128, max_seq_len=max_seq_len, temperature=1.0))

## Bonus Challenge 4: Top-k Sampling

Here's a modified version of the `generate_text` function that incorporates top-k sampling. This means that at each step, instead of considering all possible next words, we only sample from the `k` most probable words. This often helps in generating more coherent text, especially at higher temperatures.

In [ ]:
def generate_text_topk(seed_text, num_words, model, max_seq_len, temperature=1.0, top_k=None):
    """Generate `num_words` more words after `seed_text` with optional top-k sampling."""
    seed_text = clean_text(seed_text)  # apply same preprocessing as training!

    for _ in range(num_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_seq_len - 1, padding='pre')
        probs = model.predict(token_list, verbose=0)[0]

        # Apply temperature
        probs = np.log(probs + 1e-10) / temperature
        probs = np.exp(probs)

        # Apply top-k filtering if top_k is specified
        if top_k is not None and top_k > 0:
            # Get indices of top k probabilities
            top_k_indices = probs.argsort()[-top_k:]
            # Create a mask to zero out non-top-k probabilities
            mask = np.zeros_like(probs, dtype=bool)
            mask[top_k_indices] = True
            probs = np.where(mask, probs, 0.0) # Zero out non-top-k probabilities

        # Re-normalize after temperature and/or top-k filtering
        probs = probs / np.sum(probs)

        predicted_id = np.random.choice(len(probs), p=probs)

        output_word = index_to_word.get(predicted_id, "")
        if output_word == "" or output_word == "<OOV>":
            break
        seed_text += " " + output_word
    return seed_text

print("\nGenerated text with Top-K (k=10, Temperature 1.0):")
print(generate_text_topk(SEED, num_words=40, model=model, max_seq_len=max_seq_len, temperature=1.0, top_k=10))

print("\nGenerated text with Top-K (k=5, Temperature 1.5 - more constrained):")
print(generate_text_topk(SEED, num_words=40, model=model, max_seq_len=max_seq_len, temperature=1.5, top_k=5))


## Bonus Challenge 5: Number of Heads (`NUM_HEADS = 1` vs `NUM_HEADS = 8`)

We'll compare models with `NUM_HEADS = 1` and `NUM_HEADS = 8` to see how the number of attention heads affects performance and generated text, keeping `EMBED_DIM = 64` as a baseline.

### `NUM_HEADS = 1`

In [ ]:
print("--- Training with NUM_HEADS = 1 ---")

# Redefine hyperparameters
EMBED_DIM_NH = 64 # Keep embedding dimension consistent
NUM_HEADS_1 = 1
FF_DIM_NH = EMBED_DIM_NH * 2
SEQ_LEN = X.shape[1]
VOCAB_SIZE_REAL = min(len(tokenizer.word_index) + 1, VOCAB_SIZE)

# Model definition for NUM_HEADS = 1
inputs_bonus5_1 = Input(shape=(SEQ_LEN,))
x = TokenAndPositionEmbedding(SEQ_LEN, VOCAB_SIZE_REAL, EMBED_DIM_NH)(inputs_bonus5_1)
x = TransformerBlock(EMBED_DIM_NH, NUM_HEADS_1, FF_DIM_NH)(x)
x = Lambda(lambda t: t[:, -1, :])(x)
output_bonus5_1 = Dense(VOCAB_SIZE_REAL, activation="softmax")(x)

model_bonus5_1 = Model(inputs_bonus5_1, output_bonus5_1)
model_bonus5_1.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

print("Model summary for NUM_HEADS=1:")
model_bonus5_1.summary()

print("Training model with NUM_HEADS = 1...")
history_bonus5_1 = model_bonus5_1.fit(X, y, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=1)

print("\nGenerated text with NUM_HEADS = 1 (Temperature 1.0):")
print(generate_text_topk(SEED, num_words=40, model=model_bonus5_1, max_seq_len=max_seq_len, temperature=1.0))

### `NUM_HEADS = 8`

In [ ]:
print("--- Training with NUM_HEADS = 8 ---")

# Redefine hyperparameters
EMBED_DIM_NH = 64 # Keep embedding dimension consistent
NUM_HEADS_8 = 8
FF_DIM_NH = EMBED_DIM_NH * 2
SEQ_LEN = X.shape[1]
VOCAB_SIZE_REAL = min(len(tokenizer.word_index) + 1, VOCAB_SIZE)

# Model definition for NUM_HEADS = 8
inputs_bonus5_8 = Input(shape=(SEQ_LEN,))
x = TokenAndPositionEmbedding(SEQ_LEN, VOCAB_SIZE_REAL, EMBED_DIM_NH)(inputs_bonus5_8)
x = TransformerBlock(EMBED_DIM_NH, NUM_HEADS_8, FF_DIM_NH)(x)
x = Lambda(lambda t: t[:, -1, :])(x)
output_bonus5_8 = Dense(VOCAB_SIZE_REAL, activation="softmax")(x)

model_bonus5_8 = Model(inputs_bonus5_8, output_bonus5_8)
model_bonus5_8.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

print("Model summary for NUM_HEADS=8:")
model_bonus5_8.summary()

print("Training model with NUM_HEADS = 8...")
history_bonus5_8 = model_bonus5_8.fit(X, y, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=1)

print("\nGenerated text with NUM_HEADS = 8 (Temperature 1.0):")
print(generate_text_topk(SEED, num_words=40, model=model_bonus5_8, max_seq_len=max_seq_len, temperature=1.0))

## Bonus Challenge 6: Use Your Own Corpus

To use your own corpus, simply upload a `.csv` file named `tales.csv` (or any text file with a `text` column, e.g., `story.csv` with a column named `content`) into the same directory as this notebook. The setup cell in **Part 0** will automatically detect and load your custom data if it's present. Make sure your text data is in a column that can be identified (the notebook looks for 'text', 'Text', 'tale', 'story', or 'content'). Then, re-run the notebook from the top!